# Skydio X10 — RGB Orthomosaic Pipeline

Same SeaDroneLib pipeline as thermal, adapted for the Skydio RGB (MAIN_VISIBLE) sensor.

**All fixes applied from thermal pipeline validation:**
- Yaw: `VehicleOrientationNEDYaw + 90°` (same sensor body offset as thermal)
- Pitch: `90 + NED_pitch` → tilt ≈ 0° (nadir)
- FLIP_AXIS: None (flip causes 180° footprint rotation)
- ImageWidth/Height: original order

**RGB sensor specs from actual EXIF dump:**
- Resolution: 8192 x 6144 (50.3 MP)
- Focal length: 7.7 mm
- Sensor size: 12.644 x 9.483 mm
- Bands: 3 (RGB), dtype uint8

**Note:** RGB uses `georefence_images()` not `georefence_bands()`.
No Planck/Celsius calibration — RGB has no radiometric tags.

## Cell 0 — Imports

In [1]:
import os, glob, yaml, math
import rasterio
import pandas as pd
from dataclasses import asdict
from seadrone.data_structures import Profile, Partition, GeoreferencePartition, MergePartition
from seadrone.processing import FlightProcessor, Sensor
from seadrone.raster import MergeUtils
print('Imports OK')

Imports OK


## Cell 1 — Paths

**Edit only this cell.**

In [2]:
PROJECT_PATH     = r"D:\Thermal_Mosaic\RGB"
MAIN_FOLDER      = os.path.join(PROJECT_PATH, "main")          # RGB JPEGs
METADATA_CSV     = os.path.join(PROJECT_PATH, "metadata_rgb.csv")
SUMMARY_YML      = os.path.join(PROJECT_PATH, "summary_rgb.yml")
GEOREFERENCE_OUT = os.path.join(PROJECT_PATH, "georeferences", "main")
MERGE_OUT        = os.path.join(PROJECT_PATH, "merges", "main")
os.makedirs(GEOREFERENCE_OUT, exist_ok=True)
os.makedirs(MERGE_OUT, exist_ok=True)
print(f"main         : {MAIN_FOLDER}")
print(f"metadata     : {METADATA_CSV}")
print(f"summary      : {SUMMARY_YML}")
print(f"georeference : {GEOREFERENCE_OUT}")
print(f"merge        : {MERGE_OUT}")

main         : D:\Thermal_Mosaic\RGB\main
metadata     : D:\Thermal_Mosaic\RGB\metadata_rgb.csv
summary      : D:\Thermal_Mosaic\RGB\summary_rgb.yml
georeference : D:\Thermal_Mosaic\RGB\georeferences\main
merge        : D:\Thermal_Mosaic\RGB\merges\main


## Cell 2 — Sensor & Profile

Derived from actual RGB EXIF dump:  
`pixel_pitch = FocalLength_mm / CalibratedFocalLengthX_px = 7.7 / 4988.81 = 0.001543 mm/px`  
`sensor_x = 8192 × 0.001543 = 12.644 mm`,  `sensor_y = 6144 × 0.001543 = 9.483 mm`

**FLIP_AXIS = None** — same as thermal: flip causes 180° footprint rotation.

In [2]:
import math

# Current values
focal_mm  = 7.7
sensor_x  = 12.644  # mm
sensor_y  = 9.483   # mm
img_w     = 8192
img_h     = 6144
alt       = 144.0   # metres

# Computed footprint
foot_w = alt * sensor_x / focal_mm
foot_h = alt * sensor_y / focal_mm
gsd    = alt * sensor_x / (focal_mm * img_w) * 100  # cm/px

print(f"Footprint width  : {foot_w:.1f} m")
print(f"Footprint height : {foot_h:.1f} m")
print(f"GSD              : {gsd:.2f} cm/px")
print(f"Line spacing     : 20.0 m (from GPS data)")
print(f"Overlap          : {100*(1 - 20/foot_w):.0f}% side overlap")
print()
# If footprint is wrong by X%, the shoreline shifts X% of footprint width
# From the image, the shift looks like roughly 5-15m
# That's a 2-6% error in footprint → needs calibration
print("If footprint width is off by 5%:")
print(f"  Expected: {foot_w:.1f}m  vs  actual: {foot_w*0.95:.1f}m")
print(f"  Shoreline shift per line: {foot_w*0.05/2:.1f}m")

Footprint width  : 236.5 m
Footprint height : 177.3 m
GSD              : 2.89 cm/px
Line spacing     : 20.0 m (from GPS data)
Overlap          : 92% side overlap

If footprint width is off by 5%:
  Expected: 236.5m  vs  actual: 224.6m
  Shoreline shift per line: 5.9m


In [3]:
SKYDIO_RGB_SENSOR = Sensor(
    name="Skydio_VT300-L_RGB", width=8192, height=6144,
    focal_length=7.7, sensor_x=12.644, sensor_y=9.483,
    bands_number=3, band_names=["red","green","blue"],
)
# RGB_PROFILE = Profile(dtype='uint8', count=3,
#                       height=SKYDIO_RGB_SENSOR.height,
#                       width=SKYDIO_RGB_SENSOR.width, nodata=0)
# UPDATE THIS BLOCK
RGB_PROFILE = Profile(
    dtype='uint8', 
    count=3,
    height=SKYDIO_RGB_SENSOR.height,
    width=SKYDIO_RGB_SENSOR.width, 
    nodata=0,
    driver='GTiff',          # <--- ADD THIS
    # compress='lzw'           # <--- RECOMMENDED FOR 50MP FILES
)
# FLIP_AXIS = None confirmed from thermal pipeline analysis
# The +90 deg heading offset handles orientation without pixel flip
FLIP_AXIS = None

print(f"Sensor  : {SKYDIO_RGB_SENSOR.name}  {SKYDIO_RGB_SENSOR.width}x{SKYDIO_RGB_SENSOR.height}")
print(f"Profile : dtype={RGB_PROFILE.dtype}  count={RGB_PROFILE.count}")
print(f"Flip    : {FLIP_AXIS}  (None = no flip)")

Sensor  : Skydio_VT300-L_RGB  8192x6144
Profile : dtype=uint8  count=3
Flip    : None  (None = no flip)


## Cell 3 — Load Metadata & Inject Columns

`metadata_rgb.csv` has `VehicleOrientationNEDYaw` already (validated: std < 1° per line).  
Same pitch conversion as thermal. Original ImageWidth/Height order.

In [4]:
print('Loading RGB metadata ...')
processor = FlightProcessor()
flight_metadata = processor.load_metadata(
    os.path.dirname(METADATA_CSV), os.path.basename(METADATA_CSV)
)
print(f'  {len(flight_metadata)} records')

# Sensor hardware -- original order (confirmed correct from thermal analysis)
flight_metadata['FocalLength'] = SKYDIO_RGB_SENSOR.focal_length
flight_metadata['ImageWidth']  = SKYDIO_RGB_SENSOR.width    # 8192
flight_metadata['ImageHeight'] = SKYDIO_RGB_SENSOR.height   # 6144
flight_metadata['SensorX']     = SKYDIO_RGB_SENSOR.sensor_x # 12.644
flight_metadata['SensorY']     = SKYDIO_RGB_SENSOR.sensor_y # 9.483

# GPS columns
flight_metadata['GPSLatitude']  = flight_metadata['Latitude']
flight_metadata['GPSLongitude'] = flight_metadata['Longitude']
flight_metadata['GPSAltitude']  = flight_metadata['Altitude']

# Pitch conversion: same as thermal
# CameraOrientationNEDPitch ~ -89.99 → 90 + (-89.99) = 0.01 deg (nadir)
flight_metadata['Pitch'] = 90.0 + flight_metadata['Pitch']

# ID column: RGB images are JPEGs, referenced directly by filename
flight_metadata['ID'] = flight_metadata['Source']

required = ['GPSLatitude','GPSLongitude','GPSAltitude',
            'Pitch','Roll','Yaw',
            'FocalLength','ImageWidth','ImageHeight','SensorX','SensorY']
ok = all(flight_metadata[c].notna().all() for c in required)
print(f'All required columns valid: {ok}')
r0 = flight_metadata.iloc[0]
print(f'\nRow 0:')
print(f"  GPSLatitude  : {r0['GPSLatitude']:.6f}")
print(f"  GPSLongitude : {r0['GPSLongitude']:.6f}")
print(f"  Yaw          : {r0['Yaw']:.3f} deg  (VehicleOrientationNEDYaw)")
print(f"  Pitch        : {r0['Pitch']:.3f} deg  (0=nadir)")
print(f"  ID           : {r0['ID']}")

Loading RGB metadata ...
  173 records
All required columns valid: True

Row 0:
  GPSLatitude  : 33.012401
  GPSLongitude : -87.640132
  Yaw          : 61.030 deg  (VehicleOrientationNEDYaw)
  Pitch        : 0.005 deg  (0=nadir)
  ID           : S1007771.JPG


## Cell 4 — Sync Check: main/ Folder vs Metadata

In [5]:
actual_jpgs = {
    os.path.basename(p)
    for p in glob.glob(os.path.join(MAIN_FOLDER, '*.JPG'))
             + glob.glob(os.path.join(MAIN_FOLDER, '*.jpg'))
}
print(f'JPGs on disk : {len(actual_jpgs)}')
print(f'Rows in CSV  : {len(flight_metadata)}')
missing = set(flight_metadata['ID']) - actual_jpgs
if missing:
    print(f'Dropping {len(missing)} row(s) with no matching file:')
    for f in sorted(missing): print(f'  {f}')
    flight_metadata = flight_metadata[
        flight_metadata['ID'].isin(actual_jpgs)
    ].reset_index(drop=True)
else:
    print('All JPGs present ✓')
print(f'Rows ready: {len(flight_metadata)}')

JPGs on disk : 173
Rows in CSV  : 173
All JPGs present ✓
Rows ready: 173


## Cell 5 — Build Flight Lines

**+90° heading offset** — same as thermal sensor, confirmed by ArcGIS.  
The RGB and thermal sensors share the same vehicle body, so the same
`VehicleOrientationNEDYaw + 90°` correction applies to both.

In [6]:
def circular_median(angles_deg):
    sx = sum(math.sin(math.radians(a)) for a in angles_deg)
    cx = sum(math.cos(math.radians(a)) for a in angles_deg)
    return math.degrees(math.atan2(sx, cx)) % 360

print('Building flight lines ...')
with open(SUMMARY_YML) as f:
    summary = yaml.safe_load(f)

sources       = flight_metadata['Source'].tolist()
source_to_idx = {s: i for i, s in enumerate(sources)}
flight_lines  = []

for line_name, line_cfg in summary['lines'].items():
    si_img, ei_img = line_cfg['start_img'], line_cfg['end_img']
    if si_img not in source_to_idx or ei_img not in source_to_idx:
        print(f'  Skipping {line_name}: boundary image missing'); continue
    si, ei = source_to_idx[si_img], source_to_idx[ei_img]
    n = ei - si + 1
    if n < 2:
        print(f'  Skipping {line_name}: fewer than 2 images'); continue

    segment  = flight_metadata.iloc[si:ei+1]
    interior = segment.iloc[1:-1] if n > 2 else segment
    line_yaw = circular_median(interior['Yaw'].tolist())

    # +90 deg offset confirmed for thermal. Applied here too since both
    # sensors share the same vehicle body and VehicleOrientationNEDYaw.
    # If RGB tiles appear rotated in ArcGIS, test offsets 0/90/180/270
    # using the Cell 6b diagnostic and update this value.
    corrected_yaw = (line_yaw + 90.0) % 360

    # flight_lines.append({
    #     'start': si, 'end': ei + 1,
    #     'yaw':   corrected_yaw,
    #     'pitch': None, 'roll': None, 'alt': None,
    # })
    flight_lines.append({
        'start': si, 'end': ei + 1,
        'yaw':   corrected_yaw,
        'pitch': None,
        'roll':  float(interior['Roll'].median()),  # fixed median — not None'roll':  float(interior['Roll'].median()),
        'alt':   None,
    })
print(f'Built {len(flight_lines)} flight lines ✓')
for i, fl in enumerate(flight_lines):
    n = fl['end'] - fl['start']
    print(f"  Line {i+1:>2}: rows {fl['start']:>3}-{fl['end']-1:>3}  "
          f"({n} images)  yaw={fl['yaw']:.2f} deg")

Building flight lines ...
Built 15 flight lines ✓
  Line  1: rows   0- 10  (11 images)  yaw=150.92 deg
  Line  2: rows  11- 21  (11 images)  yaw=330.84 deg
  Line  3: rows  22- 32  (11 images)  yaw=150.52 deg
  Line  4: rows  33- 43  (11 images)  yaw=330.86 deg
  Line  5: rows  44- 54  (11 images)  yaw=150.71 deg
  Line  6: rows  55- 65  (11 images)  yaw=331.83 deg
  Line  7: rows  66- 76  (11 images)  yaw=151.04 deg
  Line  8: rows  77- 88  (12 images)  yaw=330.94 deg
  Line  9: rows  89-100  (12 images)  yaw=150.81 deg
  Line 10: rows 101-112  (12 images)  yaw=331.18 deg
  Line 11: rows 113-124  (12 images)  yaw=150.93 deg
  Line 12: rows 125-136  (12 images)  yaw=331.44 deg
  Line 13: rows 137-148  (12 images)  yaw=150.92 deg
  Line 14: rows 149-160  (12 images)  yaw=331.45 deg
  Line 15: rows 161-172  (12 images)  yaw=151.12 deg


## Cell 6 — Partitions

In [7]:
partitions = {
    'all':  Partition('all',  0, None, 1),
    'even': Partition('even', 0, None, 2),
    'odd':  Partition('odd',  1, None, 2),
}
PARTITIONS_TO_GEOREFERENCE = ['all']
PARTITIONS_TO_MERGE        = ['all']
print('Partitions:', list(partitions.keys()))

Partitions: ['all', 'even', 'odd']


## Cell 6b — Single-image orientation test (optional)

Run this if RGB tiles appear rotated in ArcGIS. Tests 4 heading offsets × 2
dimension orderings = 8 GeoTIFFs. Load them in ArcGIS with a basemap and
identify which filename sits correctly over the survey area.
Then update the `+90.0` offset in Cell 5 with the confirmed value.

In [19]:
import shutil
DIAG_OUT = os.path.join(PROJECT_PATH, 'orientation_test')
os.makedirs(DIAG_OUT, exist_ok=True)
from seadrone.raster import GeorefenceUtils

# Use first image of line 1
test_img = flight_metadata.iloc[0]
lat  = float(test_img['GPSLatitude'])
lon  = float(test_img['GPSLongitude'])
alt  = float(test_img['GPSAltitude'])
pitch = float(test_img['Pitch'])
roll  = float(test_img['Roll'])
line1_yaw = flight_lines[0]['yaw']

print(f'Test: {test_img["Source"]}')
print(f'GPS: lat={lat:.6f}, lon={lon:.6f}')
print(f'Line 1 yaw (with offset): {line1_yaw:.2f} deg')
print()

src_jpg = os.path.join(MAIN_FOLDER, test_img['Source'])
B = dict(lat_min=33.011, lat_max=33.016, lon_min=-87.643, lon_max=-87.637)

print(f'{"File":<50} {"TL_lat":>10} {"TL_lon":>11} {"In bounds?"}')
print('-'*80)

for yaw_offset in [0, 90, 180, 270]:
    heading = (line1_yaw + yaw_offset) % 360
    for (iw, ih, sx, sy, label) in [
        (SKYDIO_RGB_SENSOR.width,  SKYDIO_RGB_SENSOR.height,
         SKYDIO_RGB_SENSOR.sensor_x, SKYDIO_RGB_SENSOR.sensor_y, 'original'),
        (SKYDIO_RGB_SENSOR.height, SKYDIO_RGB_SENSOR.width,
         SKYDIO_RGB_SENSOR.sensor_y, SKYDIO_RGB_SENSOR.sensor_x, 'swapped'),
    ]:
        t = GeorefenceUtils.get_transform(
            f=SKYDIO_RGB_SENSOR.focal_length,
            sensor_size=(sx, sy), image_size=(iw, ih),
            lat=lat, lon=lon, alt=alt,
            yaw=heading, pitch=pitch, roll=roll
        )
        vals = [t.a,t.b,t.c,t.d,t.e,t.f]
        if any(math.isnan(v) for v in vals): continue
        corners = [t*(0,0), t*(iw,0), t*(iw,ih), t*(0,ih)]
        lats = [c[1] for c in corners]; lons = [c[0] for c in corners]
        in_b = (B['lat_min']<=min(lats) and max(lats)<=B['lat_max'] and
                B['lon_min']<=min(lons) and max(lons)<=B['lon_max'])
        fname = f'h{heading:05.1f}_off{yaw_offset:03d}_{label}.tif'
        profile_t = {'driver':'GTiff','dtype':'uint8','count':3,
                     'height':ih,'width':iw,'nodata':0,
                     'crs':'EPSG:4326','transform':t}
        import numpy as np
        with rasterio.open(src_jpg) as src:
            with rasterio.open(os.path.join(DIAG_OUT,fname),'w',**profile_t) as dst:
                dst.write(src.read()[:3])
        flag = '✓ IN BOUNDS' if in_b else '✗ OUTSIDE'
        print(f'{fname:<50} {lats[0]:>10.6f} {lons[0]:>11.6f} {flag}')

print(f'\nSaved to: {DIAG_OUT}')
print('Load in ArcGIS with basemap. Correct file = sits over survey area.')

Test: S1007771.JPG
GPS: lat=33.012401, lon=-87.640132
Line 1 yaw (with offset): 150.92 deg

File                                                   TL_lat      TL_lon In bounds?
--------------------------------------------------------------------------------


D:\Thermal_Mosaic\MosaicSeadron\seadronelib-venv\lib\site-packages\rasterio\__init__.py:321: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


h150.9_off000_original.tif                          33.012239  -87.638560 ✓ IN BOUNDS
h150.9_off000_swapped.tif                           33.011876  -87.638677 ✗ OUTSIDE
h240.9_off090_original.tif                          33.011082  -87.640325 ✓ IN BOUNDS
h240.9_off090_swapped.tif                           33.011180  -87.640758 ✓ IN BOUNDS
h330.9_off180_original.tif                          33.012563  -87.641704 ✗ OUTSIDE
h330.9_off180_swapped.tif                           33.012926  -87.641587 ✓ IN BOUNDS
h060.9_off270_original.tif                          33.013720  -87.639939 ✓ IN BOUNDS
h060.9_off270_swapped.tif                           33.013621  -87.639506 ✗ OUTSIDE

Saved to: D:\Thermal_Mosaic\RGB\orientation_test
Load in ArcGIS with basemap. Correct file = sits over survey area.


## Cell 7 — Georeference RGB Images

Uses `georefence_images()` for 3-band colour JPEGs.  
**Warning:** 8192×6144 × 173 images is large (~26 GB uncompressed). May take 30+ minutes.

In [13]:
import shutil
shutil.rmtree(GEOREFERENCE_OUT, ignore_errors=True)
os.makedirs(GEOREFERENCE_OUT, exist_ok=True)
print(f'Cleared: {GEOREFERENCE_OUT}')
print('Georeferencing RGB images ...')
print('This may take 30+ minutes for 173 × 50MP images.')

for pk in PARTITIONS_TO_GEOREFERENCE:
    part = partitions[pk]
    geo_partition = GeoreferencePartition(
        name=part.name, start=part.start, end=part.end,
        steps=part.steps, profile=RGB_PROFILE
    )
    processor.georefence_images(
        metadata=flight_metadata, partition=geo_partition,
        flight_lines=flight_lines, in_folder=MAIN_FOLDER,
        out_folder=GEOREFERENCE_OUT, flip_axis=FLIP_AXIS,
        use_metadata=True, overwrite=True,
    )

georef_files = sorted(glob.glob(
    os.path.join(GEOREFERENCE_OUT,'**','*.JPG'), recursive=True) +
    glob.glob(os.path.join(GEOREFERENCE_OUT,'**','*.tif'), recursive=True))
print(f'Georeferenced {len(georef_files)} files')

if georef_files:
    with rasterio.open(georef_files[0]) as src:
        t = src.transform
        bearing = math.degrees(math.atan2(t.a, t.d)) % 360
        corners = [t*(0,0), t*(src.width,0), t*(src.width,src.height), t*(0,src.height)]
        lats = [c[1] for c in corners]; lons = [c[0] for c in corners]
        print(f'\nSpot-check {os.path.basename(georef_files[0])}:')
        print(f'  CRS              : {src.crs}')
        print(f'  Top-edge bearing : {bearing:.1f} deg  (expect ~150 deg)')
        print(f'  Lat range        : {min(lats):.6f} to {max(lats):.6f}')
        print(f'  Lon range        : {min(lons):.6f} to {max(lons):.6f}')

Cleared: D:\Thermal_Mosaic\RGB\georeferences\main
Georeferencing RGB images ...
This may take 30+ minutes for 173 × 50MP images.


100%|████████████████████████████████████████| 173/173 [05:39<00:00,  1.96s/it]

Georeferenced 173 files

Spot-check S1007771.JPG:
  CRS              : EPSG:4326
  Top-edge bearing : 326.2 deg  (expect ~150 deg)
  Lat range        : 33.011084 to 33.013718
  Lon range        : -87.641581 to -87.638682


## georefernce image save in jpg 

In [21]:
# ==========================================
# GEOREFERENCE BLOCK (TIFF OUTPUT)
# ==========================================
import shutil
shutil.rmtree(GEOREFERENCE_OUT, ignore_errors=True)
os.makedirs(GEOREFERENCE_OUT, exist_ok=True)

print('Georeferencing RGB images (Outputting to TIFF) ...')

for pk in PARTITIONS_TO_GEOREFERENCE:
    part = partitions[pk]
    geo_partition = GeoreferencePartition(
        name=part.name, start=part.start, end=part.end,
        steps=part.steps, profile=RGB_PROFILE # This now uses the GTiff driver
    )
    processor.georefence_images(
        metadata=flight_metadata, partition=geo_partition,
        flight_lines=flight_lines, in_folder=MAIN_FOLDER,
        out_folder=GEOREFERENCE_OUT, flip_axis=FLIP_AXIS,
        use_metadata=True, overwrite=True,
    )

# Validation remains the same; it will now find the .tif files automatically
georef_files = sorted(glob.glob(os.path.join(GEOREFERENCE_OUT,'**','*.tif'), recursive=True))
print(f'Georeferenced {len(georef_files)} TIFF files')

Georeferencing RGB images (Outputting to TIFF) ...


100%|████████████████████████████████████████| 173/173 [05:01<00:00,  1.74s/it]

Georeferenced 0 TIFF files


## Histogram Matching

## Cell 8 — Merge into RGB Orthomosaic

In [14]:
import shutil
shutil.rmtree(MERGE_OUT, ignore_errors=True)
os.makedirs(MERGE_OUT, exist_ok=True)
out_mosaic = os.path.join(MERGE_OUT, 'rgb_mosaic_all_first.tif')
print('Merging ...')

try:
    for pk in PARTITIONS_TO_MERGE:
        part = partitions[pk]
        mp = MergePartition(name=part.name, start=part.start,
                            end=part.end, steps=part.steps, skip=1)
        processor.merge(
            metadata=flight_metadata, partition=mp,
            flight_lines=flight_lines, in_folder=GEOREFERENCE_OUT,
            out_folder=MERGE_OUT, method='first', band_names=None,
        )
    print('Merge succeeded ✓')
except Exception as e:
    print(f'processor.merge() failed: {e}')
    print('Falling back to MergeUtils.merge() ...')
    good = sorted(set(
        glob.glob(os.path.join(GEOREFERENCE_OUT,'**','*.JPG'),recursive=True)+
        glob.glob(os.path.join(GEOREFERENCE_OUT,'**','*.tif'),recursive=True)))
    MergeUtils.merge(raster_paths=good, out_name=out_mosaic, method='mean', band_names=None)
    print(f'Fallback succeeded ✓  → {out_mosaic}')

Merging ...


100%|██████████████████████████████████████| 173/173 [3:46:06<00:00, 78.42s/it]
D:\Thermal_Mosaic\MosaicSeadron\seadronelib-venv\lib\site-packages\numpy\core\_asarray.py:126: RuntimeWarning: invalid value encountered in cast
  arr = array(a, dtype=dtype, order=order, copy=False, subok=subok)


Merge succeeded ✓


In [1]:
import rasterio, os

tif = r"D:\Thermal_Mosaic\RGB\merges\main\rgb_mosaic_feathered.tif"
size_mb = os.path.getsize(tif) / 1e6
print(f"File size: {size_mb:.1f} MB")

with rasterio.open(tif) as src:
    print(f"CRS    : {src.crs}")
    print(f"Shape  : {src.height} x {src.width}")
    print(f"Bands  : {src.count}")
    print(f"Bounds : {src.bounds}")
    # Read a small patch to confirm data exists
    patch = src.read(1, window=rasterio.windows.Window(0, 0, 100, 100))
    print(f"Patch max value: {patch.max()}")
    print(f"File is readable: True")

File size: 1695.3 MB
CRS    : EPSG:4326
Shape  : 42391 x 38178
Bands  : 3
Bounds : BoundingBox(left=-87.64316884348354, bottom=33.0110823172077, right=-87.63728638719145, top=33.01655927174155)
Patch max value: 0
File is readable: True


## Cell 9 — Quality Check

In [ ]:
import numpy as np
mosaic_files = sorted(set(
    glob.glob(os.path.join(MERGE_OUT,'**','*.tif'),recursive=True)+
    glob.glob(os.path.join(MERGE_OUT,'**','*.TIF'),recursive=True)))
print(f'Mosaic files: {len(mosaic_files)}')
if mosaic_files:
    with rasterio.open(mosaic_files[0]) as src:
        print(f'  CRS    : {src.crs}')
        print(f'  Shape  : {src.height} x {src.width} px')
        print(f'  Bands  : {src.count}')
        print(f'  Dtype  : {src.dtypes[0]}')
        print(f'  Bounds : {src.bounds}')
        data = src.read()
        valid_mask = data[0] != 0
        print(f'  Coverage: {100*valid_mask.sum()/valid_mask.size:.1f}%')

## Cell 10 — Visualise RGB Orthomosaic

True-colour render for comparison against Agisoft output.

In [23]:
import matplotlib
if not hasattr(matplotlib.rcParams, '_get'):
    matplotlib.rcParams._get = matplotlib.rcParams.__getitem__
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np, rasterio, glob, os
from IPython.display import Image as IPImage, display

mosaic_files = sorted(glob.glob(os.path.join(MERGE_OUT,'**','*.tif'),recursive=True))
if not mosaic_files:
    print('No mosaic found — run Cells 7-8 first.')
else:
    with rasterio.open(mosaic_files[0]) as src:
        rgb    = src.read([1,2,3]).astype(np.float32)
        bounds = src.bounds

    # Normalise each band 0-1 for display
    rgb_disp = np.moveaxis(rgb, 0, -1)
    for b in range(3):
        ch = rgb_disp[:,:,b]
        valid = ch[ch > 0]
        if valid.size:
            lo, hi = np.percentile(valid, 2), np.percentile(valid, 98)
            rgb_disp[:,:,b] = np.clip((ch - lo) / max(hi-lo, 1e-6), 0, 1)
    rgb_disp[rgb_disp[:,:,0] == 0] = 0

    extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
    fig, ax = plt.subplots(figsize=(10,12), dpi=150)
    ax.imshow(rgb_disp, extent=extent, aspect='equal', origin='upper')
    ax.set_title('Skydio X10 RGB Orthomosaic', fontsize=12)
    ax.set_xlabel('Longitude (WGS84)'); ax.set_ylabel('Latitude (WGS84)')
    ax.ticklabel_format(useOffset=False)
    plt.tight_layout()
    out_png = os.path.join(MERGE_OUT, 'rgb_mosaic_preview.png')
    fig.savefig(out_png, bbox_inches='tight')
    plt.close(fig)
    display(IPImage(filename=out_png))
    print(f'Saved: {out_png}')
    print('Compare against your Agisoft RGB orthomosaic for the same area.')

KeyboardInterrupt: 